# 📈 上市公司财务分析与杜邦模型

**项目描述：** 选取A股家电行业三家龙头公司（美的、格力、海尔），基于近3年财务数据，构建杜邦分析体系、现金流预测模型与行业对标分析。

**作者：** 何易

**项目时间：** 2025.04 — 2025.06

**背景：** 全国高校商业精英挑战赛 · 会计与商业管理案例竞赛（国家级三等奖）

In [ ]:
# ==================== 1. 导入库 ====================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('✅ 库导入完成')

In [ ]:
# ==================== 2. 数据加载 ====================
df = pd.read_csv('financial_data.csv')
print(f'数据集大小: {df.shape[0]} 条, {df.shape[1]} 个字段')
print(f'覆盖公司: {df["公司"].unique()}')
print(f'覆盖年份: {df["年份"].unique()}')
df.head()

In [ ]:
# ==================== 3. 杜邦分析模型 ====================

# 杜邦分解: ROE = 净利率 × 资产周转率 × 权益乘数
# 净利率 = 净利润 / 营业收入
# 资产周转率 = 营业收入 / 总资产
# 权益乘数 = 总资产 / 净资产 = 1 / (1 - 资产负债率)

df['净利率'] = df['净利润(亿元)'] / df['营业收入(亿元)']
df['资产周转率'] = df['营业收入(亿元)'] / df['总资产(亿元)']
df['权益乘数'] = df['总资产(亿元)'] / df['净资产(亿元)']
df['杜邦ROE'] = df['净利率'] * df['资产周转率'] * df['权益乘数']

print('杜邦分析结果:')
display_cols = ['公司','年份','净利率','资产周转率','权益乘数','杜邦ROE','ROE(%)']
print(df[display_cols].to_string(index=False))

# 精度检查
df['ROE_diff'] = abs(df['杜邦ROE'] - df['ROE(%)']/100)
print(f'\n杜邦ROE与实际ROE平均误差: {df["ROE_diff"].mean()*100:.4f}%')

In [ ]:
# ==================== 4. ROE驱动因素可视化 ====================

# 按公司分组计算3年均值
avg_dupond = df.groupby('公司')[['净利率','资产周转率','权益乘数','杜邦ROE']].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
companies = df['公司'].unique()
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

# 4.1 净利率对比
axes[0,0].bar(avg_dupond.index, avg_dupond['净利率']*100, color=colors)
axes[0,0].set_title('平均净利率对比(%)')
axes[0,0].set_ylabel('净利率(%)')
for i, v in enumerate(avg_dupond['净利率']*100):
    axes[0,0].text(i, v+0.2, f'{v:.2f}%', ha='center')

# 4.2 资产周转率对比
axes[0,1].bar(avg_dupond.index, avg_dupond['资产周转率'], color=colors)
axes[0,1].set_title('平均资产周转率对比')
axes[0,1].set_ylabel('周转率(次)')
for i, v in enumerate(avg_dupond['资产周转率']):
    axes[0,1].text(i, v+0.01, f'{v:.3f}', ha='center')

# 4.3 权益乘数对比
axes[1,0].bar(avg_dupond.index, avg_dupond['权益乘数'], color=colors)
axes[1,0].set_title('平均权益乘数对比（杠杆水平）')
axes[1,0].set_ylabel('权益乘数')
for i, v in enumerate(avg_dupond['权益乘数']):
    axes[1,0].text(i, v+0.05, f'{v:.2f}', ha='center')

# 4.4 综合ROE对比
axes[1,1].bar(avg_dupond.index, avg_dupond['杜邦ROE']*100, color=colors)
axes[1,1].set_title('综合ROE对比(%)')
axes[1,1].set_ylabel('ROE(%)')
for i, v in enumerate(avg_dupond['杜邦ROE']*100):
    axes[1,1].text(i, v+0.3, f'{v:.2f}%', ha='center')

plt.tight_layout()
plt.savefig('dupont_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== 5. 财务趋势分析 ====================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics = [
    ('营业收入(亿元)', '营业收入趋势', '亿元'),
    ('净利润(亿元)', '净利润趋势', '亿元'),
    ('毛利率(%)', '毛利率趋势', '%'),
    ('净利率(%)', '净利率趋势', '%'),
    ('ROE(%)', 'ROE趋势', '%'),
    ('资产负债率(%)', '资产负债率趋势', '%')
]

for idx, (metric, title, unit) in enumerate(metrics):
    row, col = divmod(idx, 3)
    ax = axes[row, col]
    for i, company in enumerate(companies):
        subset = df[df['公司'] == company]
        ax.plot(subset['年份'], subset[metric], marker='o', label=company, 
                color=colors[i], linewidth=2, markersize=8)
        for _, row_data in subset.iterrows():
            val = f'{row_data[metric]:.1f}' if unit != '%' else f'{row_data[metric]:.1f}'
            ax.annotate(val, (row_data['年份'], row_data[metric]), 
                       textcoords='offset points', xytext=(0,10), ha='center', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(unit)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('financial_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== 6. 现金流预测 ====================
# 使用线性回归预测2025年经营活动现金流

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

future_year = 2025
predictions = {}

for i, company in enumerate(companies):
    subset = df[df['公司'] == company].copy()
    
    X = subset['年份'].values.reshape(-1, 1)
    y = subset['经营活动现金流(亿元)'].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    # 预测
    pred = model.predict([[future_year]])[0]
    predictions[company] = pred
    
    # 历史 + 预测
    years = list(subset['年份']) + [future_year]
    cf_values = list(subset['经营活动现金流(亿元)']) + [pred]
    
    ax = axes[i]
    ax.plot(years[:-1], cf_values[:-1], marker='o', color=colors[i], 
            linewidth=2, markersize=8, label='历史')
    ax.plot([years[-2], years[-1]], [cf_values[-2], cf_values[-1]], 
            marker='s', color='orange', linewidth=2, linestyle='--', markersize=8, label='预测')
    ax.set_title(f'{company} 现金流预测', fontsize=12)
    ax.set_ylabel('经营活动现金流(亿元)')
    ax.set_xlabel('年份')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xticks(years)
    
    # 标注预测值
    ax.annotate(f'{pred:.0f}亿', (future_year, pred),
               textcoords='offset points', xytext=(0,15), ha='center',
               fontsize=11, fontweight='bold', color='orange')

plt.tight_layout()
plt.savefig('cashflow_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 2025年经营活动现金流预测:')
for company, pred in predictions.items():
    print(f'  {company}: {pred:.1f} 亿元')

In [ ]:
# ==================== 7. 行业对标雷达图 ====================
from math import pi

# 选择5个关键指标
radar_metrics = ['毛利率(%)','净利率(%)','ROE(%)','资产周转率','经营活动现金流(亿元)']
radar_df = avg_dupond.copy()

# 获取各公司雷达数据
company_values = {}
for company in companies:
    subset = df[df['公司'] == company]
    vals = []
    for m in radar_metrics:
        vals.append(subset[m].mean())
    company_values[company] = vals

# 创建雷达图
N = len(radar_metrics)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'polar': True})

for i, company in enumerate(companies):
    values = company_values[company]
    # 归一化
    max_vals = [max([company_values[c][j] for c in companies]) for j in range(N)]
    normalized = [values[j]/max_vals[j] if max_vals[j] > 0 else 0 for j in range(N)]
    normalized += normalized[:1]
    
    ax.plot(angles, normalized, 'o-', linewidth=2, label=company, color=colors[i])
    ax.fill(angles, normalized, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, fontsize=11)
ax.set_title('三家公司财务行业对标雷达图', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.tight_layout()
plt.savefig('radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== 8. 投资价值评估总结 ====================
print('='*60)
print('📋 投资价值评估总结')
print('='*60)

last_year = df[df['年份'] == df['年份'].max()]

for i, company in enumerate(companies):
    row = last_year[last_year['公司'] == company].iloc[0]
    print(f'\n--- {company} ({row["年份"]}年) ---')
    print(f'  ROE: {row["ROE(%)"]:.2f}%')
    print(f'  净利率: {row["净利率(%)中"] if "净利率(%)中" in row else row["净利率(%)"]:.2f}%' if '净利率(%)' in last_year else f'  净利率: {(row["净利润(亿元)"]/row["营业收入(亿元)"]*100):.2f}%')
    print(f'  经营现金流: {row["经营活动现金流(亿元)"]:.1f}亿元')
    print(f'  资产负债率: {row["资产负债率(%)"]:.2f}%')

print()
print('建议:')
print('  1. 格力净利率领先，盈利能力最强')
print('  2. 美的营收规模最大，资产周转率最高')
print('  3. 海尔现金流稳健，财务结构相对保守')
print('  4. 三家公司ROE均>14%，整体盈利能力优秀')